# Lewisham · `decision_date` (web: "Decision date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `decision_date` for every Lewisham application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY` (the format the schema
  and the index mapping give).

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "decision_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "decision_date"])
print("Lewisham applications:", len(df))

Lewisham applications: 29021


In [3]:
value = df["decision_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,1725,5.94
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,27296,94.06
total,29021,100.00


Values that are not dates:


,id,decision_date


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records:

| | Count | Share |
|---|---|---|
| null | 1,725 | 5.94% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 27,296 | 94.06% |

- **Clean format:** every filled value is a real `DD/MM/YYYY` date.
- **1,725 have no decision_date** (Brent: 995). The 1,151 empty records are among them by
  definition; the rest is not broken down yet.

## Step 2 · Records with no decision_date, by status and decision

Aggregations on `status.raw` × `decision.raw` for records with no `decision_date`
(`missing` shown as "(none)"). The empty records (no status and no decision here) are
part of the "(none)" × "(none)" cell.

In [4]:
body = {
    "size": 0,
    "query": {"bool": {"filter": [lewisham_filter()], "must_not": [{"exists": {"field": "decision_date"}}]}},
    "aggs": {"status": {"terms": {"field": "status.raw", "size": 100, "missing": "(none)"},
                        "aggs": {"decision": {"terms": {"field": "decision.raw", "size": 100, "missing": "(none)"}}}}},
}
buckets = search(body)["aggregations"]["status"]["buckets"]
nodate = pd.DataFrame([(s["key"], d["key"], d["doc_count"]) for s in buckets for d in s["decision"]["buckets"]],
                      columns=["status", "decision", "count"])
print("No decision_date:", nodate["count"].sum())
nodate.sort_values("count", ascending=False, ignore_index=True)

No decision_date: 1725


,status,decision,count
0,(none),(none),1156
1,Application Under Consideration,(none),553
2,Application Invalid,(none),8
3,(none),Withdrawn,5
4,Application Invalid,Approved,1
5,Application Invalid,Refused,1
6,Completed,Approved,1


In [5]:
# The records with a decision but no decision_date (other than the empty-status Withdrawn ones).
FIELDS = ["id", "status", "decision", "valid_date", "decision_target_date", "bo_system", "last_updated", "description"]
body = {"size": 20, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"exists": {"field": "decision.raw"}}, {"exists": {"field": "status.raw"}}],
                           "must_not": [{"exists": {"field": "decision_date"}}]}}}
listed = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
listed["description"] = listed["description"].str.slice(0, 60)
listed

,id,status,decision,valid_date,decision_target_date,bo_system,last_updated,description
0,Lewisham-DC_21_124764,Completed,Approved,14/12/2021,None,CSV Data Import,2025-08-14T12:44:17.805+01:00,"Prior Approval under Schedule 2, Part 3, Class..."
1,Lewisham-TEST_18_022222,Application Invalid,Refused,22/09/2026,None,CSV Data Import,2026-11-10T10:00:02.785Z,Application for a Lawful Development Certifica...
2,Lewisham-TEST_69_022223,Application Invalid,Approved,28/09/2026,None,CSV Data Import,2026-11-10T10:00:15.192Z,Test Case to illustrate the Golden RulesTEST C...


### Findings · Step 2

| status | decision | Count | What they are |
|---|---|---|---|
| (none) | (none) | 1,156 | 1,151 empty records (dropped) + 5 with only a valid_date (4 dropped as portal-removed; DC/24/138293 Unknown) |
| Application Under Consideration | (none) | 553 | pending (1 is TEST/13/022223/X, dropped) |
| Application Invalid | (none) | 8 | invalid (1 is TEST/80/022222/X, dropped) |
| (none) | Withdrawn | 5 | the DC/ records whose status is copied from decision; withdrawn with no date |
| Application Invalid | Approved / Refused | 2 | TEST/69/022223 and TEST/18/022222 (dropped) |
| Completed | Approved | 1 | Lewisham-DC_21_124764 |

- **Almost all as expected:** after cleaning, records with no decision_date are pending
  (552), invalid (7), withdrawn with no date (5), DC/24/138293 (Unknown) and one other.
- **One decided record with no date:** Lewisham-DC_21_124764, a Prior Approval valid
  14/12/2021, Approved and Completed, no decision_date. Brent had 18 decided records
  without a date; Lewisham has 1 (after the test records go).
- After cleaning: 552 + 7 + 5 + 1 + 1 = **566 records with no decision_date.**

## Step 3 · Type check and most common values

The Python type of every value in `df` (Step 1). Then the range, the most common dates
(to spot placeholder dates such as Brent's early-1987 month-only dates), the share on the
1st of a month, and the count by year.

In [6]:
types = df["decision_date"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Types:", types.value_counts().to_dict())

filled = df["decision_date"].dropna()
dates = pd.to_datetime(filled, format="%d/%m/%Y")
print("Earliest:", dates.min().date(), "| latest:", dates.max().date(),
      "| after today (2026-11-11):", int((dates > "2026-11-11").sum()))
print("On the 1st of a month:", int((dates.dt.day == 1).sum()), f"({100 * (dates.dt.day == 1).mean():.1f}%; 1/30 = 3.3%)")
print()
print("Most common decision dates:")
print(filled.value_counts().head(10).to_string())
print()
print("By year:")
print(dates.dt.year.value_counts().sort_index().to_string())

Types: {'str': 27296, 'NoneType': 1725}
Earliest: 1987-01-12 | latest: 2026-11-14 | after today (2026-11-11): 1
On the 1st of a month: 831 (3.0%; 1/30 = 3.3%)

Most common decision dates:
decision_date
14/09/2023    45
20/12/2024    42
16/06/2022    40
28/04/2022    35
12/07/2019    33
18/04/2019    33
18/05/2022    33
03/03/2023    32
06/10/2022    32
13/09/2022    31

By year:
decision_date
1987      37
1988      36
1989      41
1990      43
1991      34
1992      32
1993      40
1994      33
1995      34
1996      35
1997      23
1998      22
1999      28
2000     144
2001     204
2002     186
2003     275
2004     256
2005     278
2006     237
2007     297
2008     276
2009     183
2010     198
2011     205
2012     150
2013     103
2014     167
2015     168
2016     177
2017     165
2018     191
2019    2897
2020    2985
2021    3642
2022    3356
2023    3094
2024    2686
2025    2439
2026    1899


In [7]:
# The decision date after today.
future_ids = df.loc[pd.to_datetime(df["decision_date"], format="%d/%m/%Y") > "2026-11-11", "id"].tolist()
body = {"size": 5, "_source": ["id", "status", "decision", "decision_date", "valid_date", "last_updated", "description"],
        "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": future_ids}}]}}}
pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]]).T

,0
decision_date,14/11/2026
last_updated,2026-11-10T10:03:28.110Z
valid_date,14/09/2026
decision,Approved
description,Schedule 1 application for the construction of...
id,Lewisham-DC_26_145383
status,Approved


### Findings · Step 3

- **Strings or null only** (27,296 strings, 1,725 null); no numbers.
- **No placeholder dates:** the most common date (14/09/2023) has 45 records; 3.0% of
  dates fall on the 1st of a month, about what chance gives (3.3%). Brent's early-1987
  month-only dates have no counterpart (1987 has 37 decisions in all).
- **One date in the future:** Lewisham-DC_26_145383, Approved with decision_date
  14/11/2026 (3 days after today), valid 14/09/2026, last_updated 2026-11-10. Not
  explained (recorded ahead of issue?).
- **Thin before 2019:** 1987–1999 have 22–43 decisions a year, 2000–2018 have 103–297;
  from 2019 on, 1,899–3,642 a year. Before 2019 there are 4,298 decisions in all, about
  as many as the 4,312 legacy records (not matched record by record). So counts by year
  before 2019 are far from complete.

## Step 4 · `decision_date` after cleaning

The drops and the status and decision rules of `semantic/lewisham/cleaning.yaml` applied
by hand (as in the status notebook, Step 5), then the counts `decision_date.yaml` gives.

In [8]:
import yaml

rules = yaml.safe_load((PROJECT_ROOT / "semantic/lewisham/cleaning.yaml").read_text())
COLS = ["id", "lpa_app_no", "status", "decision", "decision_date", "valid_date", "decision_target_date",
        "description", "bo_system"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]
raw = pd.DataFrame(rows).reindex(columns=COLS)
for col in COLS[1:]:
    raw[col] = raw[col].map(lambda v: v.strip() or None if isinstance(v, str) else None)

keep = pd.Series(True, index=raw.index)
for rule in rules["all_records"]:
    if rule["action"] == "drop_records":
        hit = raw["id"].isin(rule.get("ids", []))
        if "when_all_null" in rule:
            hit |= raw[rule["when_all_null"]].isna().all(axis=1)
        keep &= ~hit
c = raw[keep].copy()
portal = next(r for r in rules["all_records"] if r["id"] == "outcome_from_portal")
fill = c["id"].isin(portal["outcomes"]) & c["status"].isna() & c["decision"].isna()
for col in ["status", "decision"]:
    c.loc[fill, col] = c.loc[fill, "id"].map(portal["outcomes"])
for rule in rules["fields"]["status"]:
    if rule["action"] == "map":
        c.loc[c["status"] == rule["from"], "status"] = rule["to"]
    elif rule["action"] == "copy_when_null":
        hit = c["status"].isna() & c["decision"].notna()
        c.loc[hit, "status"] = c.loc[hit, "decision"]
    elif rule["action"] == "fill_null":
        c.loc[c["status"].isna(), "status"] = rule["to"]
for rule in rules["fields"]["decision"]:
    c.loc[c["decision"] == rule["from"], "decision"] = rule["to"]
for col in ["decision_date", "valid_date", "decision_target_date"]:
    c[col + "_d"] = pd.to_datetime(c[col], format="%d/%m/%Y", errors="coerce")
print("Records after cleaning:", len(c))

nodate = c[c["decision_date_d"].isna()]
print("No decision_date:", len(nodate), "| by status:", nodate["status"].value_counts().to_dict())
print("Decided (decision set) but no decision_date:", nodate.loc[nodate["decision"].notna() & (nodate["status"] != "Withdrawn"), ["id", "status", "decision"]].to_dict("records"))
print("Withdrawn with no date:", int((nodate["status"] == "Withdrawn").sum()))
print("Future decision_date (after 2026-11-11):", c.loc[c["decision_date_d"] > "2026-11-11", "id"].tolist())
print("decision_date before valid_date:", int((c["decision_date_d"] < c["valid_date_d"]).sum()))
years = c["decision_date_d"].dt.year
print("Decisions before 2019:", int((years < 2019).sum()), "| per year 2019-2026:", years[years >= 2019].value_counts().sort_index().to_dict())
print("Earliest / latest:", c["decision_date_d"].min().date(), c["decision_date_d"].max().date())

Records after cleaning: 27765
No decision_date: 495 | by status: {'Application Under Consideration': 481, 'Application Invalid': 7, 'Withdrawn': 5, 'Completed': 1, 'Unknown': 1}
Decided (decision set) but no decision_date: [{'id': 'Lewisham-DC_21_124764', 'status': 'Completed', 'decision': 'Approved'}]
Withdrawn with no date: 5
Future decision_date (after 2026-11-11): ['Lewisham-DC_26_145383']
decision_date before valid_date: 14
Decisions before 2019: 4298 | per year 2019-2026: {2019.0: 2893, 2020.0: 2983, 2021.0: 3640, 2022.0: 3350, 2023.0: 3087, 2024.0: 2684, 2025.0: 2437, 2026.0: 1898}
Earliest / latest: 1987-01-12 2026-11-14


### Findings · Step 4

27,765 records after cleaning:
- **No decision_date: 495** (not 566): Application Under Consideration 481, Application
  Invalid 7, Withdrawn 5, Unknown 1 (DC/24/138293), and DC/21/124764 (Completed, Approved).
- Unchanged: 1 future decision_date (DC/26/145383), 14 with decision_date before
  valid_date, 4,298 decisions before 2019, range 12/01/1987 to 14/11/2026.
- 2019–2026: 1,898–3,640 decisions a year.